# Tai Dinh, Week 7, Advanced Models

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn import linear_model as lm
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# Load in CSV's, Remove Outliers, and Select Features

In [ ]:
test_df = pd.read_csv('/content/test_df.csv')
valid_df = pd.read_csv('/content/valid_df.csv')
train_df = pd.read_csv('/content/train_df.csv')

/tmp/ipykernel_1361/319966960.py:1: DtypeWarning: Columns (118,175) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv('/content/test_df.csv')
/tmp/ipykernel_1361/319966960.py:2: DtypeWarning: Columns (118) have mixed types. Specify dtype option on import or set low_memory=False.
  valid_df = pd.read_csv('/content/valid_df.csv')
/tmp/ipykernel_1361/319966960.py:3: DtypeWarning: Columns (118,175) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('/content/train_df.csv')


In [ ]:
train_df = train_df[(train_df["ClosePrice"] <= 600000000) & (train_df["ClosePrice"] > 0)]

In [ ]:
cols = ["LivingArea", "Age", "Bed/Bath Ratio", "Living Area per Bedroom", "ParkingTotal", "BathroomsTotalInteger", "GarageSpaces", "AttachedGarageYN", "PoolPrivateYN", "ViewYN", "BedroomsTotal", "Stories", "AssociationFee"
, "AssociationFeeFrequency_Monthly", "AssociationFeeFrequency_None", "AssociationFeeFrequency_Quarterly", "AssociationFeeFrequency_SemiAnnually"
, 'CountyOrParish_Amador',
 'CountyOrParish_Butte',
 'CountyOrParish_Calaveras',
 'CountyOrParish_Clark',
 'CountyOrParish_Colusa',
 'CountyOrParish_Contra Costa',
 'CountyOrParish_El Dorado',
 'CountyOrParish_Foreign Country',
 'CountyOrParish_Fresno',
 'CountyOrParish_Glenn',
 'CountyOrParish_Humboldt',
 'CountyOrParish_Imperial',
 'CountyOrParish_Inyo',
 'CountyOrParish_Kern',
 'CountyOrParish_Kings',
 'CountyOrParish_Lake',
 'CountyOrParish_Lassen',
 'CountyOrParish_Los Angeles',
 'CountyOrParish_Madera',
 'CountyOrParish_Marin',
 'CountyOrParish_Mariposa',
 'CountyOrParish_Mendocino',
 'CountyOrParish_Merced',
 'CountyOrParish_Modoc',
 'CountyOrParish_Mono',
 'CountyOrParish_Monterey',
 'CountyOrParish_Napa',
 'CountyOrParish_Nevada',
 'CountyOrParish_Orange',
 'CountyOrParish_Other',
 'CountyOrParish_Other State',
 'CountyOrParish_Placer',
 'CountyOrParish_Plumas',
 'CountyOrParish_Riverside',
 'CountyOrParish_Sacramento',
 'CountyOrParish_San Benito',
 'CountyOrParish_San Bernardino',
 'CountyOrParish_San Diego',
 'CountyOrParish_San Francisco',
 'CountyOrParish_San Joaquin',
 'CountyOrParish_San Luis Obispo',
 'CountyOrParish_San Mateo',
 'CountyOrParish_Santa Barbara',
 'CountyOrParish_Santa Clara',
 'CountyOrParish_Santa Cruz',
 'CountyOrParish_Shasta',
 'CountyOrParish_Sierra',
 'CountyOrParish_Siskiyou',
 'CountyOrParish_Solano',
 'CountyOrParish_Sonoma',
 'CountyOrParish_Stanislaus',
 'CountyOrParish_Sutter',
 'CountyOrParish_Tehama',
 'CountyOrParish_Trinity',
 'CountyOrParish_Tulare',
 'CountyOrParish_Tuolumne',
 'CountyOrParish_Ventura',
 'CountyOrParish_Yolo',
 'CountyOrParish_Yuba',
 'DistrictName_Apple Valley Unified',
 'DistrictName_Bear Valley Unified',
 'DistrictName_Beaumont Unified',
 'DistrictName_Capistrano Unified',
 'DistrictName_Chico Unified',
 'DistrictName_Chino Valley Unified',
 'DistrictName_Coachella Valley Unified',
 'DistrictName_Conejo Valley Unified',
 'DistrictName_Corona-Norco Unified',
 'DistrictName_Desert Sands Unified',
 'DistrictName_Fontana Unified',
 'DistrictName_Fremont Unified',
 'DistrictName_Garden Grove Unified',
 'DistrictName_Glendale Unified',
 'DistrictName_Hemet Unified',
 'DistrictName_Hesperia Unified',
 'DistrictName_Irvine Unified',
 'DistrictName_Lake Elsinore Unified',
 'DistrictName_Long Beach Unified',
 'DistrictName_Los Angeles Unified',
 'DistrictName_Lucia Mar Unified',
 'DistrictName_Moreno Valley Unified',
 'DistrictName_Morongo Unified',
 'DistrictName_Mt. Diablo Unified',
 'DistrictName_Murrieta Valley Unified',
 'DistrictName_Newport-Mesa Unified',
 'DistrictName_None',
 'DistrictName_Oakland Unified',
 'DistrictName_Oceanside Unified',
 'DistrictName_Orange Unified',
 'DistrictName_Other',
 'DistrictName_Palm Springs Unified',
 'DistrictName_Pasadena Unified',
 'DistrictName_Placentia-Yorba Linda Unified',
 'DistrictName_Poway Unified',
 'DistrictName_Redlands Unified',
 'DistrictName_Rim of the World Unified',
 'DistrictName_Riverside Unified',
 'DistrictName_Saddleback Valley Unified',
 'DistrictName_San Bernardino City Unified',
 'DistrictName_San Diego Unified',
 'DistrictName_San Jose Unified',
 'DistrictName_San Marcos Unified',
 'DistrictName_San Ramon Valley Unified',
 'DistrictName_Simi Valley Unified',
 'DistrictName_Temecula Valley Unified',
 'DistrictName_Torrance Unified',
 'DistrictName_Vista Unified',
 'DistrictName_West Contra Costa Unified']
X_train = train_df[cols]
Y_train = train_df["ClosePrice"]

X_valid = valid_df[cols]
Y_valid = valid_df["ClosePrice"]

X_test = test_df[cols]
Y_test = test_df["ClosePrice"]

In [ ]:
train_df.shape

(118076, 245)

#Gradient Boosting
Here I'm experimenting with parameters to find the best $r^2$ on the validation set

In [ ]:
gradmodel = xgb.XGBRegressor(objective='reg:squarederror',
    n_estimators=80,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='rmse')
gradmodel.fit(X_train, Y_train)
y_pred = gradmodel.predict(X_valid)

print(r2_score(Y_valid, y_pred))
print(root_mean_squared_error(Y_valid, y_pred))

0.4134440215035242
1283933.6142724077


With optimal parameters, try xgboost on test set:

In [ ]:
test_pred = gradmodel.predict(X_test)
print(r2_score(Y_test, test_pred))
print(root_mean_squared_error(Y_test, test_pred))

0.6602658312184324
895097.045340494


XGBoost seems to improve on random forests by a bit for the test set!